In [ ]:
#This is a parameters cell.  Any args passed in from the driver notebook will overwrite these defaults

xmla_endpoint = None #None = Current Workspace's XMLA Endpoint
workspace = "Fabric Load Test"
perf_analyzer_filename = '/lakehouse/default/Files/PerfScenarios/Queries/PowerBIPerformanceData.json'
model = "DIAD Final Report with RLS"
roles = None  #Used with customdata to force active roles.  Not needed with effective_username
customdata = "foo" #customdata for use in RLS without impersonation
effective_username = None #for RLS with impersionation.  Must be the UPN of a user with read+build for the model
iterations = 3  #number of times to run the perf_analyzer_filename in this scenario
delay_sec = 1  #number of seconds to wait between iterations
loadtestId = "localtesting" # name of load test for logging
threadId = 1 #id of this virtual user to seperate logging for multiuser-testing
concurrent_threads = 1  #number of concurrent threads in this test; passed in here only for logging
load_test_name = "localtesting" #name of the test (prefix of loadtestId), for logging

# --- Think times (disabled by default: behaviour identical to the original notebook) ---
enable_think_times = False #replay the pauses between user actions recorded in the performance analyzer file
think_time_scale = 1.0 #1.0 = recorded pace; 0.5 = twice as fast; 2.0 = twice as slow; 0.0 = no internal pauses
max_think_time_sec = 60.0 #cap applied to any single pause
think_time_jitter_pct = 0.0 #0.10 = +/-10% random variation on each pause
fallback_think_time_sec = 0.0 #pause between queries when the file has no User Action events
start_delay_sec = 0.0 #ramp-up: seconds this virtual user waits before starting (computed by RunLoadTest)
initial_user_jitter_sec = 0.0 #extra random wait (0..N seconds) before the first action
stop_on_query_error = False #True = the virtual user stops at its first failed query (behaviour before 3.4.0); the error row and the CSV are kept


In [ ]:
# Think-time helpers (standard library only; unit-tested outside Fabric)
import bisect
import json
import random
import time
from datetime import datetime, timezone


def _parse_ts(value):
    """ISO-8601 -> epoch seconds. Returns None when the value is not a valid timestamp."""
    if not value or not isinstance(value, str):
        return None
    try:
        text = value.strip()
        if text.endswith("Z"):
            text = text[:-1] + "+00:00"
        parsed = datetime.fromisoformat(text)
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        return parsed.timestamp()
    except (ValueError, OverflowError, OSError):
        return None


def validate_think_time_params(scale, maximum_sec, jitter_pct, ramp_up_sec=0.0, initial_jitter_sec=0.0, fallback_sec=0.0):
    if scale < 0:
        raise ValueError("think_time_scale cannot be negative")
    if maximum_sec < 0:
        raise ValueError("max_think_time_sec cannot be negative")
    if not 0.0 <= jitter_pct <= 1.0:
        raise ValueError("think_time_jitter_pct must be between 0.0 and 1.0")
    if ramp_up_sec < 0 or initial_jitter_sec < 0 or fallback_sec < 0:
        raise ValueError("ramp_up_sec, initial_user_jitter_sec and fallback_think_time_sec cannot be negative")


def calculate_applied_think_time(
    original_think_time_sec: float,
    enabled: bool,
    scale: float,
    maximum_sec: float,
    jitter_pct: float = 0.0,
) -> float:
    if not enabled:
        return 0.0

    original = max(0.0, float(original_think_time_sec or 0.0))
    scaled = min(original * scale, maximum_sec)

    if jitter_pct > 0.0 and scaled > 0.0:
        scaled *= random.uniform(1.0 - jitter_pct, 1.0 + jitter_pct)

    return max(0.0, scaled)


def load_queries(fn: str, group_by_action: bool = False, fallback_think_time_sec: float = 0.0) -> list:
    """Load the DAX queries of a Power BI Performance Analyzer export.

    Returns a flat list. Every query keeps its original fields and gains the fields of the user action
    ('User Action' event) it belongs to. Actions are detected from the 'User Action' events, whatever their
    sourceLabel is; a query belongs to the last action that started before it. When group_by_action is False the
    file order is kept (original behaviour); when True the list is stably sorted by action.
    """
    with open(fn, encoding="utf-8-sig") as f:
        events = json.loads(f.read())["events"]

    queries = []
    visual_name = ""
    visual_id = ""
    for e in events:
        name = e.get("name")
        if name == "Visual Container Lifecycle":
            visual_name = e["metrics"]["visualTitle"]
            visual_id = e["metrics"]["visualId"]
        if name == "Execute DAX Query":
            queries.append({
                "visual_name": visual_name,
                "visual_id": visual_id,
                "query_text": e["metrics"]["QueryText"],
                "_start": _parse_ts(e.get("start")),
                "_end": _parse_ts(e.get("end")),
            })

    actions = []
    for e in events:
        if e.get("name") == "User Action":
            start = _parse_ts(e.get("start"))
            if start is not None:
                actions.append({
                    "start": start,
                    "end": _parse_ts(e.get("end")),
                    "label": (e.get("metrics") or {}).get("sourceLabel") or "UserAction",
                })
    actions.sort(key=lambda a: a["start"])
    starts = [a["start"] for a in actions]

    # action_index: 1..N following the chronological order of the User Action events; 0 = before the first action
    last_index = 0
    for position, q in enumerate(queries, start=1):
        if not actions:
            index = position
        elif q["_start"] is not None:
            index = bisect.bisect_right(starts, q["_start"])
        else:
            index = last_index
        q["action_index"] = index
        last_index = index

    groups = {}
    for q in queries:
        groups.setdefault(q["action_index"], []).append(q)

    previous_group_end = None
    previous_group_start = None
    for index in sorted(groups):
        members = groups[index]
        if not actions:
            label, action_start, action_end = "NoUserAction", members[0]["_start"], None
        elif index == 0:
            label, action_start, action_end = "PreAction", members[0]["_start"], None
        else:
            action = actions[index - 1]
            label, action_start, action_end = action["label"], action["start"], action["end"]

        query_ends = [m["_end"] if m["_end"] is not None else m["_start"] for m in members]
        query_ends = [x for x in query_ends if x is not None]
        group_end = action_end if action_end is not None else (max(query_ends) if query_ends else None)

        if previous_group_start is None:
            think, basis = 0.0, "first_action"
        elif not actions:
            think, basis = float(fallback_think_time_sec or 0.0), "fallback"
        elif action_start is not None and previous_group_end is not None:
            think, basis = max(0.0, action_start - previous_group_end), "previous_group_end"
        elif action_start is not None:
            think, basis = max(0.0, action_start - previous_group_start), "start_to_start"
        else:
            think, basis = float(fallback_think_time_sec or 0.0), "fallback"

        for order, m in enumerate(members):
            m["action_label"] = label
            m["action_query_count"] = len(members)
            m["think_time_before_sec"] = round(think, 3)
            m["think_time_basis"] = basis
            m["is_first_query_in_action"] = order == 0

        previous_group_start = action_start if action_start is not None else (members[0]["_start"] or 0.0)
        previous_group_end = group_end

    if group_by_action:
        queries = sorted(queries, key=lambda q: q["action_index"])
    for q in queries:
        q.pop("_start", None)
        q.pop("_end", None)
    return queries


In [ ]:
import pandas
from typing import Iterable
import json
import time
import sempy.fabric as fabric
tom = fabric.create_tom_server() #get CLR loaded
from datetime import datetime


from Microsoft.AnalysisServices.AdomdClient import AdomdConnection

def run_query(con: AdomdConnection, query:str) -> int:
    cmd = con.CreateCommand()
    cmd.CommandText = query
    rows = 0
    rdr = cmd.ExecuteReader()
    try:
        while rdr.Read():
            rows=rows+1
    finally:
        rdr.Close()  # an open reader would block the next query of this connection
    return rows

# load_queries() is defined in the think-time helpers cell

think_time_log = []  # one row per applied pause; written to logs/<loadtestId>/thinktimes/
query_error_count = 0  # failed queries of this virtual user (each one is also a row with status 'error')


def _redact(text: str) -> str:
    access_token = globals().get("token")
    return text.replace(access_token, "<redacted-token>") if access_token else text


def _record_query_error(con: AdomdConnection, message: str) -> None:
    """Log a failed query and reopen the connection if the error closed it; the virtual user carries on."""
    global query_error_count
    query_error_count += 1
    with open(f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}/error.log", "a") as file:
        file.write(f"user {threadId} query error: {message}\n")
    try:
        state = getattr(con, "State", None)
        if state is not None and str(state) not in ("Open", "1"):  # ConnectionState.Open
            con.Close()
            con.Open()
    except Exception as reopen_error:  # the next queries will fail and be recorded as well
        print(f"Virtual user {threadId}: could not reopen the connection: {_redact(str(reopen_error))[:200]}")


def run_perf_scenario(con: AdomdConnection, queries: Iterable[str], i: int, sink: list = None) -> list:
    # Rows are appended to `sink` as they happen, so they survive an error later in the iteration
    results = sink if sink is not None else []
    # With think times enabled, the pauses between actions replace the per-query wait;
    # delay_sec still applies between complete iterations (see the main cell).
    per_query_delay_sec = 0 if enable_think_times else delay_sec
    for qn, q in enumerate(queries, start=1):
        original_think_time_sec = 0.0
        scaled_think_time_sec = 0.0
        applied_think_time_sec = 0.0

        if enable_think_times and q["is_first_query_in_action"]:
            original_think_time_sec = q["think_time_before_sec"]
            scaled_think_time_sec = min(max(0.0, original_think_time_sec) * think_time_scale, max_think_time_sec)
            applied_think_time_sec = calculate_applied_think_time(
                original_think_time_sec=original_think_time_sec,
                enabled=enable_think_times,
                scale=think_time_scale,
                maximum_sec=max_think_time_sec,
                jitter_pct=think_time_jitter_pct,
            )
            sleep_start_utc = datetime.now(timezone.utc)
            if applied_think_time_sec > 0:
                time.sleep(applied_think_time_sec)
            sleep_end_utc = datetime.now(timezone.utc)
            think_time_log.append({
                "load_test_name": load_test_name,
                "loadtest_id": loadtestId,
                "thread_id": threadId,
                "effective_username": effective_username,
                "iteration": i,
                "action_index": q["action_index"],
                "source_label": q["action_label"],
                "think_time_basis": q["think_time_basis"],
                "original_think_time_sec": round(original_think_time_sec, 3),
                "think_time_scale": think_time_scale,
                "scaled_think_time_sec": round(scaled_think_time_sec, 3),
                "applied_think_time_sec": round(applied_think_time_sec, 3),
                "sleep_start_utc": sleep_start_utc.isoformat(),
                "sleep_end_utc": sleep_end_utc.isoformat(),
                "queries_in_action": q["action_query_count"],
            })

        start = time.time()
        query_text = q["query_text"]
        visual_name = q["visual_name"]
        visual_id = q["visual_id"]
        status, error = "ok", ""
        try:
            rows = run_query(con, query_text)
        except Exception as query_error:  # a failed query (timeout, throttling, RLS...) is a result, not the end of the user
            rows, status = 0, "error"
            error = _redact(f"{type(query_error).__name__}: {query_error}").replace("\r", " ").replace("\n", " ")[:500]
            _record_query_error(con, error)
        duration = time.time() - start

        result = {
            "loadtest_id": loadtestId,
            "model": model,
            "concurrent_threads": concurrent_threads,
            "iterations": iterations,
            "delay_sec": delay_sec,
            "query_number": qn,
            "visual_name": visual_name,
            "visual_id": visual_id,
            "iteration": i,
            "query": query_text,
            "rows": rows,
            "duration": duration,
            "start_time": start,
            "start_time_dt": datetime.fromtimestamp(start),
            "customdata": customdata,
            "effective_username": effective_username,
            "thread_id": threadId,
            "action_index": q["action_index"],
            "source_label": q["action_label"],
            "original_think_time_sec": round(original_think_time_sec, 3),
            "scaled_think_time_sec": round(scaled_think_time_sec, 3),
            "applied_think_time_sec": round(applied_think_time_sec, 3),
            "status": status,
            "error": error,
        }
        results.append(result)
        if status == "error" and str(stop_on_query_error).strip().lower() in ("true", "1", "yes"):
            raise RuntimeError(f"Virtual user {threadId} stopped at query {qn} (stop_on_query_error): {error}")
        time.sleep(per_query_delay_sec)
    return results


In [ ]:
import sempy.fabric as fabric
import pandas 
import csv
import uuid
tom = fabric.create_tom_server() #get CLR loaded
from Microsoft.AnalysisServices.AdomdClient import AdomdConnection 


# --- Think times and staggered start (the defaults reproduce the original behaviour) ---
def _as_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


enable_think_times = _as_bool(enable_think_times)
think_time_scale = float(think_time_scale)
max_think_time_sec = float(max_think_time_sec)
think_time_jitter_pct = float(think_time_jitter_pct)
fallback_think_time_sec = float(fallback_think_time_sec)
start_delay_sec = float(start_delay_sec)
initial_user_jitter_sec = float(initial_user_jitter_sec)
validate_think_time_params(think_time_scale, max_think_time_sec, think_time_jitter_pct,
                           start_delay_sec, initial_user_jitter_sec, fallback_think_time_sec)

initial_wait_sec = start_delay_sec + (random.uniform(0.0, initial_user_jitter_sec) if initial_user_jitter_sec > 0 else 0.0)
if initial_wait_sec > 0:
    print(f"Virtual user {threadId}: staggered start, waiting {initial_wait_sec:.1f}s")
    _wait_start = datetime.now(timezone.utc)
    time.sleep(initial_wait_sec)
    think_time_log.append({
        "load_test_name": load_test_name, "loadtest_id": loadtestId, "thread_id": threadId,
        "effective_username": effective_username, "iteration": -1, "action_index": 0,
        "source_label": "initial_offset", "think_time_basis": "initial_offset",
        "original_think_time_sec": 0.0, "think_time_scale": 1.0,
        "scaled_think_time_sec": round(initial_wait_sec, 3), "applied_think_time_sec": round(initial_wait_sec, 3),
        "sleep_start_utc": _wait_start.isoformat(), "sleep_end_utc": datetime.now(timezone.utc).isoformat(),
        "queries_in_action": 0,
    })

token = notebookutils.credentials.getToken("pbi")
notebookutils.fs.mkdirs(f'/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}') #This should already exist, but useful here when running in standalong mode

# if (xmla_endpoint == None):
#     xmla_endpoint = f"powerbi://api.powerbi.com/v1.0/myorg/{notebookutils.runtime.context['currentWorkspaceName']}"
xmla_endpoint = f"powerbi://api.powerbi.com/v1.0/myorg/{workspace}"


constr = f"Data Source={xmla_endpoint};Initial Catalog={model};Timeout=7200;"
if (effective_username != None):
    constr = constr + f"EffectiveUserName={effective_username};"
if (customdata != None):
    constr = constr + f"CustomData={customdata};"
    # constr = constr + f"CustomData=;"
if (roles != None):
     constr = constr + f"Roles={roles};"

safe_constr = constr.replace(token, "<redacted-token>") if token else constr  # never log the access token


def open_adomd_connection(connection_string, access_token):
    """ADOMD connection authenticated like semantic-link (sempy): AccessToken plus a refresh callback.

    Some tenants reject the access token passed as a password in the connection string ("Authentication failed
    for all authenticators") while sempy's token is accepted. Falls back to the password form without sempy.
    """
    try:
        from functools import partial
        from Microsoft.AnalysisServices import AccessToken
        from System import Func
        # Private sempy module (the same helper semantic-link uses for TOM). A move is caught by the ImportError
        # fallback below; a change of signature is not, so re-test this after semantic-link upgrades.
        from sempy.fabric._client._utils import refresh_tom_access_token
    except ImportError:
        return AdomdConnection(connection_string + f"password={access_token};")
    connection = AdomdConnection(connection_string)
    connection.AccessToken = refresh_tom_access_token(connection.AccessToken, None)
    connection.OnAccessTokenExpired = Func[AccessToken, AccessToken](
        partial(refresh_tom_access_token, credential=None))
    return connection
print(f"The connection string is :{safe_constr}")
con = open_adomd_connection(constr, token)
all_results = []  # filled while the queries run, written in `finally` even if the virtual user stops early
filename = f"{loadtestId}_user_{threadId}"
try:
    con.Open()

    queries = load_queries(perf_analyzer_filename, group_by_action=enable_think_times,
                           fallback_think_time_sec=fallback_think_time_sec)
    if not queries:
        raise ValueError(f"{perf_analyzer_filename} contains no 'Execute DAX Query' events")
    if enable_think_times:
        _actions = [q for q in queries if q["is_first_query_in_action"]]
        print(f"Think times enabled: {len(_actions)} actions, {len(queries)} queries, "
              f"scale={think_time_scale}, max={max_think_time_sec}s, jitter={think_time_jitter_pct:.0%}")

    print(f"starting {iterations} iterations with {delay_sec} think time between")
    for i in range(iterations):
        #print(f"Starting iteration {i}")
        run_perf_scenario(con, queries, i, sink=all_results)
        if (i<iterations):
            print(f"Completed iteration, starting {delay_sec}sec think time")
            time.sleep(delay_sec)
    if query_error_count:
        print(f"Virtual user {threadId}: {query_error_count} of {len(all_results)} queries failed (see error.log)")

    
    con.Close()

except Exception as e:
    print(e)
    with open(f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}/error.log", 'a') as file:
         file.write(f'{safe_constr}  : {e}\n')
    raise
finally:
    try:
        # Write results to CSV files in lakehouse for analyzing later (also the partial results of a user that failed)
        if all_results:
            with open(f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}/{filename}.csv", 'w', newline='') as file:
                writer = csv.DictWriter(file, fieldnames=all_results[0].keys())
                writer.writeheader()
                writer.writerows(all_results)

        if think_time_log:
            think_dir = f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}/thinktimes"  # sub-folder: kept apart from the result CSV files
            notebookutils.fs.mkdirs(think_dir)
            think_fields = list({key: None for row in think_time_log for key in row})
            with open(f"{think_dir}/{filename}_thinktimes.csv", 'w', newline='') as file:
                writer = csv.DictWriter(file, fieldnames=think_fields)
                writer.writeheader()
                writer.writerows(think_time_log)
    except Exception as write_error:
        print(f"Virtual user {threadId}: could not write its results: {write_error}")
